# PPO: Clipped Surrogate Objective

*Builds on `docs/policy_gradient.md` §7 and `docs/ppo.md`.*

Why vanilla policy gradient diverges, and how PPO's clipped surrogate fixes it.

In [ ]:
import sys, os
from pathlib import Path
_p = Path.cwd()
while _p != _p.parent and not (_p / 'utils' / 'ppo.py').exists():
    _p = _p.parent
REPO = str(_p)
sys.path.insert(0, REPO)

from utils.env     import ENV
from utils.toy_mdp import GridWorldEnv, TabularPolicy, TabularValue, reinforce
from utils.ppo     import NeuralPolicy, NeuralValue, ppo, ppo_clip_loss
from utils.train   import plot_history, plot_histories, smooth

import torch, numpy as np, matplotlib.pyplot as plt
ENV.summary()

## 1. Vanilla PG: Importance Ratio Explosion

Vanilla PG updates on the *current* policy's samples. If we reuse old samples (off-policy), we need importance sampling:

$$\mathbb{E}_{a \sim \pi_{\theta_{\text{old}}}}\!\left[\frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} A(s,a)\right]$$

The ratio $r_t(\theta) = \pi_\theta / \pi_{\theta_{\text{old}}}$ is unbounded — a large update can make $r_t \gg 1$, producing an enormous gradient step and destabilising training.

In [ ]:
# Demonstrate ratio explosion: scale a fixed perturbation by δ,
# old policy makes one action rare so the ratio for that action can blow up.
torch.manual_seed(0)
n_actions:  int          = 4
logits_old: torch.Tensor = torch.tensor([0., 0., 0., -8.])  # action 3 rare under old
direction:  torch.Tensor = torch.tensor([0., 0., 0., 1.])   # δ lifts action 3
deltas:     torch.Tensor = torch.linspace(0, 8.0, 40)

kl_vals:     list[float] = []
ratio_maxes: list[float] = []
for delta in deltas:
    logits_new = logits_old + delta * direction
    p_old = torch.softmax(logits_old, dim=-1)
    p_new = torch.softmax(logits_new, dim=-1)
    ratios = p_new / p_old
    kl_vals.append((p_old * (p_old.log() - p_new.log())).sum().item())
    ratio_maxes.append(ratios.max().item())

plt.figure(figsize=(7, 4))
plt.plot(kl_vals, ratio_maxes, 'o-', ms=4)
plt.axhline(1.2, color='red', ls='--', label='clip boundary (ε=0.2)')
for i in range(0, len(deltas), 5):
    plt.annotate(f'δ={deltas[i]:.2f}',
                 xy=(kl_vals[i], ratio_maxes[i]),
                 xytext=(5, 5), textcoords='offset points', fontsize=8)
plt.xlabel('KL(old || new)')
plt.ylabel('max importance ratio')
plt.title('Importance ratio grows with policy change (δ scales the perturbation)')
plt.legend(); plt.tight_layout(); plt.show()

## 2. The Clipped Surrogate (docs/ppo.md §3)

PPO clips the ratio to $[1-\varepsilon, 1+\varepsilon]$:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(r_t A_t,\; \text{clip}(r_t, 1{-}\varepsilon, 1{+}\varepsilon) A_t\right)\right]$$

Taking the *min* ensures we never benefit from moving $r_t$ outside the clip range:
- If $A_t > 0$: gradient zeroed when $r_t > 1+\varepsilon$ (policy already improved enough)
- If $A_t < 0$: gradient zeroed when $r_t < 1-\varepsilon$ (policy already pushed down enough)

In [ ]:
# Visualise L^CLIP as a function of r_t for fixed advantage
r = torch.linspace(0.5, 1.8, 200)
eps = 0.2

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, A in zip(axes, [1.0, -1.0]):
    unclipped = r * A
    clipped   = torch.clamp(r, 1-eps, 1+eps) * A
    objective = torch.min(unclipped, clipped)
    ax.plot(r, unclipped.numpy(), '--', label='$r_t A_t$ (unclipped)', alpha=0.7)
    ax.plot(r, clipped.numpy(),   '--', label='clip$(r_t)\cdot A_t$', alpha=0.7)
    ax.plot(r, objective.numpy(), 'k-', lw=2,  label='$L^{CLIP}$ (min)')
    ax.axvspan(1-eps, 1+eps, alpha=0.1, color='green', label='trust region')
    ax.axvline(1.0, color='gray', ls=':')
    ax.set_xlabel('$r_t$'); ax.set_ylabel('objective')
    ax.set_title(f'A = {A:+.1f}')
    ax.legend(fontsize=8)
plt.suptitle('PPO clipped surrogate', fontsize=12)
plt.tight_layout(); plt.show()

## 3. PPO vs REINFORCE on GridWorld

Both trained for equivalent environment steps. PPO uses a neural policy (MLP) and collects batches of 512 steps before updating, whereas REINFORCE updates episode-by-episode.

In [ ]:
# ── REINFORCE: tabular policy on 5×5 GridWorld ──
torch.manual_seed(0)
env_re    = GridWorldEnv(H=5, W=5, goal=(4,4), start=(0,0), gamma=0.99, max_steps=100)
policy_re = TabularPolicy(n_states=env_re.n_states, n_actions=env_re.n_actions)
value_re  = TabularValue(n_states=env_re.n_states)

history_re = reinforce(env_re, policy_re, value_re,
                       n_episodes=4000,
                       lr_policy=5e-2,
                       lr_value=5e-2,
                       gamma=0.99)
print(f"REINFORCE mean return (last 200 eps): "
      f"{np.mean(history_re['returns'][-200:]):.4f}")

# ── PPO: neural policy on 4×4 GridWorld ──
torch.manual_seed(0)
env_ppo    = GridWorldEnv(H=4, W=4, goal=(3,3), start=(0,0), gamma=0.99, max_steps=100)
policy_ppo = NeuralPolicy(n_states=env_ppo.n_states, n_actions=env_ppo.n_actions)
value_ppo  = NeuralValue(n_states=env_ppo.n_states)

history_ppo = ppo(env_ppo, policy_ppo, value_ppo,
                  n_iterations=150, n_steps=512, n_epochs=4,
                  lr=3e-4, epsilon=0.2)
print(f"PPO mean return (last 20 iters): "
      f"{np.mean(history_ppo['returns'][-20:]):.4f}")

In [ ]:
# Overlay returns, pg_loss, and ratio_mean across both algorithms.
# x is normalized to training progress since REINFORCE measures episodes
# and PPO measures iterations. ratio_mean exists only for PPO and is
# silently skipped for REINFORCE.
plot_histories(
    {'REINFORCE (5×5 tabular)': history_re,
     'PPO (4×4 neural)':        history_ppo},
    keys=['returns', 'pg_loss', 'ratio_mean'],
    smooth_window=20,
    title='REINFORCE vs PPO',
)

## 4. Effect of Clipping Threshold $\varepsilon$

Small $\varepsilon$: very conservative updates, slow learning.  
Large $\varepsilon$: approaches unclipped importance sampling, risks instability.

In [ ]:
results = {}
for eps in [0.05, 0.2, 0.5]:
    torch.manual_seed(0)
    env2 = GridWorldEnv(H=4,W=4,goal=(3,3),start=(0,0),gamma=0.99,max_steps=100)
    p = NeuralPolicy(env2.n_states, env2.n_actions)
    v = NeuralValue(env2.n_states)
    h = ppo(env2, p, v, n_iterations=150, n_steps=512, n_epochs=4,
            lr=3e-4, epsilon=eps)
    results[eps] = h['returns']

from utils.train import smooth
plt.figure(figsize=(7,3.5))
for eps, rets in results.items():
    plt.plot(smooth(rets, 15), label=f'ε={eps}')
plt.xlabel('iteration'); plt.ylabel('mean return')
plt.title('PPO: effect of clipping threshold ε')
plt.legend(); plt.tight_layout(); plt.show()

## 5. Summary

The PPO surrogate at a single timestep is

$$L_t \;=\; r_t \cdot A_t, \qquad r_t = \frac{\pi_{\text{new}}(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$$

so $|L_t|$ blows up through *two multiplicative factors*:

1. **Ratio explosion** — $\pi_{\text{old}}(a_t)$ small (action rare under old) while $\pi_{\text{new}}(a_t)$ is not small (the new policy lifted it). The importance-sampling correction is extrapolating outside the regime where old gave us evidence.
2. **Advantage magnitude** — $|A_t|$ large means this step matters a lot, regardless of policy distance.

Two distinct regimes:

- **Justified big update.** High $A_t$ on an action the old policy *already sampled often enough* — the advantage estimate is trustworthy. The update should be taken; PPO still takes it, just spread across more iterations rather than one giant step.
- **Spurious gradient explosion.** The action was rare under old, the new policy upweighted it, and the advantage estimate comes from a sparse, high-variance observation. Without clipping, a single rare sample's noisy advantage gets multiplied by a huge ratio and dominates the batch gradient — a big step taken on unreliable evidence.

**Clipping bounds the multiplier, not the signal:**

$$L_t^{\text{PPO}} = \min\!\Bigl(r_t A_t,\ \operatorname{clip}(r_t, 1-\varepsilon, 1+\varepsilon)\cdot A_t\Bigr)$$

The advantage $A_t$ is left untouched — we still respond to genuine signal. Only the importance-correction multiplier $r_t$ is capped at $1\pm\varepsilon$, so any single step's contribution is bounded by $\sim(1+\varepsilon)|A_t|$. Real high-advantage updates still happen; they just can't be amplified arbitrarily by a runaway off-policy correction. The `min` asymmetry handles signs: for $A_t>0$ the upper clip $1+\varepsilon$ binds, for $A_t<0$ the lower clip $1-\varepsilon$ binds — always the more *pessimistic* of the two estimates.